# **SQL_Task**

## Setup — create an in-memory database

In [15]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

def run_query(query):
    """Helper: run a SELECT and return a pandas DataFrame for clean display."""
    return pd.read_sql_query(query, conn)

setup_sql = """
-- General-purpose tables (used for the "usual" questions)
CREATE TABLE employees (
    emp_id     INTEGER PRIMARY KEY,
    name       TEXT NOT NULL,
    department TEXT NOT NULL,
    salary     INTEGER NOT NULL
);

CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    name        TEXT NOT NULL,
    city        TEXT NOT NULL
);

CREATE TABLE orders (
    order_id    INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    amount      INTEGER NOT NULL,
    order_date  TEXT NOT NULL,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
);

CREATE TABLE students (
    student_id INTEGER PRIMARY KEY,
    name       TEXT NOT NULL,
    subject    TEXT NOT NULL,
    marks      INTEGER NOT NULL
);

-- Project-related tables (used for the dashboard questions)
CREATE TABLE devices (
    device_id   INTEGER PRIMARY KEY,
    device_name TEXT NOT NULL,
    location    TEXT NOT NULL
);

CREATE TABLE alerts (
    alert_id       INTEGER PRIMARY KEY,
    device_id      INTEGER NOT NULL,
    source_ip      TEXT NOT NULL,
    attack_type    TEXT NOT NULL,
    severity       TEXT NOT NULL,
    status         TEXT NOT NULL DEFAULT 'Open',
    created_at     TEXT NOT NULL,
    FOREIGN KEY (device_id) REFERENCES devices(device_id)
);

-- Sample data
INSERT INTO employees VALUES
(1, 'Aditi Sharma', 'IT', 55000),
(2, 'Rohan Mehta', 'IT', 48000),
(3, 'Priya Nair', 'HR', 42000),
(4, 'Karan Singh', 'Sales', 60000),
(5, 'Neha Verma', 'Sales', 39000),
(6, 'Aman Gupta', 'IT', 70000);

INSERT INTO customers VALUES
(1, 'Ravi Kumar', 'Mumbai'),
(2, 'Sneha Rao', 'Pune'),
(3, 'Vikram Joshi', 'Mumbai'),
(4, 'Anjali Desai', 'Delhi');

INSERT INTO orders VALUES
(101, 1, 2500, '2026-06-01'),
(102, 1, 1500, '2026-06-10'),
(103, 2, 4200, '2026-06-05'),
(104, 3, 800,  '2026-06-15'),
(105, 4, 3000, '2026-06-20'),
(106, 2, 1000, '2026-06-22');

INSERT INTO students VALUES
(1, 'Aditi', 'Math', 88),
(2, 'Aditi', 'Science', 92),
(3, 'Rohan', 'Math', 65),
(4, 'Rohan', 'Science', 70),
(5, 'Priya', 'Math', 95),
(6, 'Priya', 'Science', 91),
(7, 'Priya', 'English', 89);

INSERT INTO devices VALUES
(1, 'WEB-SRV-01', 'DMZ'),
(2, 'DB-SRV-01', 'Internal'),
(3, 'HR-LAPTOP-07', 'Office-Floor2'),
(4, 'VPN-GATEWAY', 'Edge');

INSERT INTO alerts VALUES
(1001, 1, '45.33.10.7',   'ddos',        'High',     'Open',     '2026-07-01 09:15:00'),
(1002, 2, '192.168.1.55', 'brute_force', 'Medium',   'Open',     '2026-07-01 10:02:00'),
(1003, 1, '45.33.10.7',   'port_scan',   'Low',      'Resolved', '2026-07-01 11:47:00'),
(1004, 3, '10.0.0.99',    'malware',     'Critical', 'Resolved', '2026-07-02 08:20:00'),
(1005, 4, '203.0.113.5',  'brute_force', 'Medium',   'Open',     '2026-07-02 09:05:00'),
(1006, 1, '45.33.10.7',   'ddos',        'High',     'Open',     '2026-07-03 02:11:00'),
(1007, 2, '198.51.100.2', 'malware',     'Critical', 'Resolved', '2026-07-03 12:55:00'),
(1008, 1, '45.33.10.7',   'ddos',        'High',     'Open',     '2026-07-04 01:30:00');
"""

cursor.executescript(setup_sql)
conn.commit()
print("Database ready. Tables: employees, customers, orders, students, devices, alerts")


Database ready. Tables: employees, customers, orders, students, devices, alerts


# **Question 1:**  
Find all employees who earn more than the average salary across
the whole company

In [13]:
%%sql
INSERT INTO devices (device_id, device_name, ip_address, location) VALUES
(1, 'WEB-SRV-01', '192.168.1.10', 'DMZ'),
(2, 'DB-SRV-01',  '192.168.1.20', 'Internal'),
(3, 'HR-LAPTOP-07','10.0.0.55',   'Office-Floor2'),
(4, 'VPN-GATEWAY', '10.0.0.1',    'Edge');

INSERT INTO users (user_id, full_name, role) VALUES
(1, 'Aditi Sharma', 'Analyst'),
(2, 'Rohan Mehta',  'Analyst'),
(3, 'Priya Nair',   'Admin');

INSERT INTO alerts (alert_id, device_id, resolved_by, attack_type, severity, severity_score, created_at, status) VALUES
(101, 1, 1,    'ddos',        'High',     8, '2026-07-01 09:15:00', 'Resolved'),
(102, 2, NULL, 'brute_force', 'Medium',   5, '2026-07-01 10:02:00', 'Open'),
(103, 1, NULL, 'port_scan',   'Low',      2, '2026-07-01 11:47:00', 'Open'),
(104, 3, 2,    'malware',     'Critical', 9, '2026-07-02 08:20:00', 'Resolved'),
(105, 4, NULL, 'brute_force', 'Medium',   6, '2026-07-02 09:05:00', 'Open'),
(106, 1, 1,    'phishing',    'Medium',   4, '2026-07-02 14:30:00', 'Resolved'),
(107, 2, NULL, 'ddos',        'High',     7, '2026-07-03 02:11:00', 'Open'),
(108, 3, NULL, 'port_scan',   'Low',      3, '2026-07-03 06:40:00', 'Open'),
(109, 4, 3,    'malware',     'Critical', 10,'2026-07-03 12:55:00', 'Resolved'),
(110, 1, NULL, 'ddos',        'High',     8, '2026-07-04 01:30:00', 'Open');


 * sqlite:///security.db
4 rows affected.
3 rows affected.
10 rows affected.


[]

# **Question 2 (General — JOIN + GROUP BY + LIMIT)**
Find the top 3 customers by total amount spent, showing their
name, city, and total spend.

In [16]:
run_query("""
SELECT c.name, c.city, SUM(o.amount) AS total_spent
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
GROUP BY c.customer_id
ORDER BY total_spent DESC
LIMIT 3;
""")


,name,city,total_spent
0,Sneha Rao,Pune,5200
1,Ravi Kumar,Mumbai,4000
2,Anjali Desai,Delhi,3000


# Question 3 (General — GROUP BY + HAVING)
Find students who scored above 85 marks in more than one
subject.

In [17]:
run_query("""
SELECT name, COUNT(*) AS subjects_above_85
FROM students
WHERE marks > 85
GROUP BY name
HAVING COUNT(*) > 1;
""")


,name,subjects_above_85
0,Aditi,2
1,Priya,3


# Question 4 (General — classic "second highest" question)
Find the department-wise highest salary, and separately, find
the second-highest salary in the whole company (a very common interview-style
SQL question).

In [18]:
print("Highest salary per department:")
display(run_query("""
SELECT department, MAX(salary) AS highest_salary
FROM employees
GROUP BY department;
"""))

print("\nSecond-highest salary in the company:")
display(run_query("""
SELECT MAX(salary) AS second_highest_salary
FROM employees
WHERE salary < (SELECT MAX(salary) FROM employees);
"""))

Highest salary per department:


,department,highest_salary
0,HR,42000
1,IT,70000
2,Sales,60000



Second-highest salary in the company:


,second_highest_salary
0,60000


# Question 5 (General — classic "find duplicates" question)
Assume `customers` could contain duplicate city entries by
mistake. Write a query to find any city names that appear more than once in
the customers table (a common "find duplicates" interview question).

In [19]:
run_query("""
SELECT city, COUNT(*) AS occurrences
FROM customers
GROUP BY city
HAVING COUNT(*) > 1;
""")


,city,occurrences
0,Mumbai,2


## Question 6 (Project — aggregation)
Count how many alerts exist for each severity level, ordered
from most to least frequent.

In [20]:
run_query("""
SELECT severity, COUNT(*) AS alert_count
FROM alerts
GROUP BY severity
ORDER BY alert_count DESC;
""")


,severity,alert_count
0,High,3
1,Medium,2
2,Critical,2
3,Low,1


## Question 7 (Project — JOIN + HAVING)
Find devices that have generated more than 2 alerts in total,
showing device name, location, and alert count.

In [21]:
run_query("""
SELECT d.device_name, d.location, COUNT(a.alert_id) AS alert_count
FROM alerts a
JOIN devices d ON a.device_id = d.device_id
GROUP BY d.device_id
HAVING COUNT(a.alert_id) > 2
ORDER BY alert_count DESC;
""")


,device_name,location,alert_count
0,WEB-SRV-01,DMZ,4


## Question 8 (Project — top offenders)
Find the top 3 source IP addresses responsible for the most
alerts overall.

In [22]:
run_query("""
SELECT source_ip, COUNT(*) AS total_alerts
FROM alerts
GROUP BY source_ip
ORDER BY total_alerts DESC
LIMIT 3;
""")


,source_ip,total_alerts
0,45.33.10.7,4
1,203.0.113.5,1
2,198.51.100.2,1


## Question 9 (Project — CASE + aggregation)
For each device, show the percentage of its alerts that are
currently "Resolved" versus "Open" (use `CASE` inside an aggregate).

In [23]:
run_query("""
SELECT
    d.device_name,
    COUNT(a.alert_id) AS total_alerts,
    ROUND(100.0 * SUM(CASE WHEN a.status = 'Resolved' THEN 1 ELSE 0 END) / COUNT(a.alert_id), 1) AS percent_resolved,
    ROUND(100.0 * SUM(CASE WHEN a.status = 'Open' THEN 1 ELSE 0 END) / COUNT(a.alert_id), 1) AS percent_open
FROM alerts a
JOIN devices d ON a.device_id = d.device_id
GROUP BY d.device_id
ORDER BY percent_open DESC;
""")


,device_name,total_alerts,percent_resolved,percent_open
0,VPN-GATEWAY,1,0.0,100.0
1,WEB-SRV-01,4,25.0,75.0
2,DB-SRV-01,2,50.0,50.0
3,HR-LAPTOP-07,1,100.0,0.0


## Question 10
List every student's average marks across all subjects, and
separately, list every device's average severity_score-equivalent by
counting how many "High"/"Critical" alerts (the higher-risk ones) it has —
practicing the same "average per group" pattern in two different contexts.

In [24]:
print("Average marks per student:")
display(run_query("""
SELECT name, ROUND(AVG(marks), 1) AS average_marks
FROM students
GROUP BY name
ORDER BY average_marks DESC;
"""))

print("\nHigh/Critical alert count per device:")
display(run_query("""
SELECT d.device_name, COUNT(*) AS high_or_critical_alerts
FROM alerts a
JOIN devices d ON a.device_id = d.device_id
WHERE a.severity IN ('High', 'Critical')
GROUP BY d.device_id
ORDER BY high_or_critical_alerts DESC;
"""))


Average marks per student:


,name,average_marks
0,Priya,91.7
1,Aditi,90.0
2,Rohan,67.5



High/Critical alert count per device:


,device_name,high_or_critical_alerts
0,WEB-SRV-01,3
1,HR-LAPTOP-07,1
2,DB-SRV-01,1
